[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module3_DeepLearning/17_PyTorch_NN.ipynb)

# MATH 232 - Math Models w/ Tech
## PyTorch & Deep Learning Intro
### Instructor: Prof. Mario Bañuelos 

In [6]:
# Import necessary packages
from IPython.display import HTML, Image
import numpy as np
import pandas as pd
# import plotting packages
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import torch
import torch.nn as nn
import torch.nn.functional as F

## Outline

* [Deep Learning & Neural Networks](#nn)
 * [Creating Models / Lin Regression](#linreg)
* [Further Reading](#reading)

# What is Deep Learning? <a id='nn'></a>

**Deep Learning** is a subset of machine learning that are mainly based on artificial neural networks. Learning can be

* supervised
* *semi-supervised*
* unsupervised

## Neural Networks

**Neural networks** were first proposed in 1944 by W. Mccullough and W. Pitts.

* Their popularity has ebbed and flowed over the years, and has had a resurgence due to the improvement of computational advances.

* The first trainable model was proposed by Rosenblatt in 1957.

![](https://docs.google.com/uc?export=download&id=1UYmUw_YSUP0I4duuN4X3X54NuWTDrG7t)

![](https://docs.google.com/uc?export=download&id=1J2zxCiRcVb3dtlqNY0X-UiZXv4f4PtBn)

![](https://docs.google.com/uc?export=download&id=1t2ARdywKIlKrEiKtEHYxPTZM4w6TEpxa)

![](https://docs.google.com/uc?export=download&id=122kjrXTMPyaXtsOdgDYRUde8IIbVfn_P)

Here, $H$ is the Heaviside function,

$$
H(z) = \begin{cases} 0 \text{ if } z \le 0 \\ 1 \text{ if } z > 1 \end{cases}
$$

## Stacking Perceptrons

* If we only have one layer  $\rightarrow$  linear reg.
* Let’s stack multiple perceptrons.
* Output layer incorporates a nonlinear transformation (known as activation functions) 

* Technically, when the number of hidden layers > 2, we call it deep learning.
* There is no general rule for how deep or large a neural network should be.

![simple neural network](https://upload.wikimedia.org/wikipedia/commons/e/e4/Artificial_neural_network.svg)

## Terminology 

* Feed-Forward Network
* Input Layer
* Hidden Layers
* Output Layer
* Weights & Biases
* Loss Function
* Learning Rate
* Activation Function

### Practice (10-15 min)
 - Navigate to this <a href="http://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=xor&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=3&seed=0.80789&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false"> online neural network applet. </a>
 - In your groups, choose one type of data and explore how these different parts affect the test loss and classificaiton:
  * Number of Hidden Layers
  * Number of Neurons
  * Learning Rate
  * Activation Function (time permitting)
  
* *Note*: You start by pressing the play button and you may press stop after 1000 epochs (which is closely related to the number of iterations).

## Simplified GD Loop

In [3]:
# Here we generate some fake data
def lin(a,b,x): return a*x+b

def gen_fake_data(n, a, b):
    x = np.random.uniform(0,1,n) 
    y = lin(a,b,x) + 0.1 * np.random.normal(0,3,n)
    return x, y

In [7]:
# linear tranformation with input dimension=1 and output dimension=1
nn.Linear(1, 1)

Linear(in_features=1, out_features=1, bias=True)

### Models in Pytorch

In [8]:
# simple way of specifying a linear regression model
model = torch.nn.Sequential(
    # take 1 input, and output 1
    nn.Linear(1, 1),
)
model

Sequential(
  (0): Linear(in_features=1, out_features=1, bias=True)
)

In [10]:
# note here we have just two parameters, why?
print([p for p in model.parameters()])

[Parameter containing:
tensor([[0.0626]], requires_grad=True), Parameter containing:
tensor([-0.5517], requires_grad=True)]


In [11]:
x, y = gen_fake_data(10000, 3., 8.)
x = torch.tensor(x).float()
y = torch.tensor(y).float()
x.shape

torch.Size([10000])

In [12]:
# you have to be careful with the dimensions that your model is expecting
x1 = torch.unsqueeze(x, 1)
x1.shape

torch.Size([10000, 1])

In [13]:
y_hat = model(x1)
print(y_hat)

tensor([[-0.5302],
        [-0.5391],
        [-0.5339],
        ...,
        [-0.5161],
        [-0.5306],
        [-0.5234]], grad_fn=<AddmmBackward>)


In [14]:
# Use the optim package to define an Optimizer that will update the weights of
# the model for us. Here we will use Adam
learning_rate = 0.1
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [15]:
for t in range(10000):
    # Forward pass: compute predicted y using operations on Variables
    y_hat = model(x1)
    loss = F.mse_loss(y_hat, y.unsqueeze(1))
    if t % 1000 == 0: print(loss.item())
       
    # Before the backward pass, use the optimizer object to zero all of the
    # gradients for the variables
    optimizer.zero_grad()
    loss.backward()
    
    # Calling the step function on an Optimizer makes an update to its
    # parameters
    optimizer.step()

101.33794403076172
0.09001843631267548
0.08999752998352051
0.08999751508235931
0.08999750763177872
0.08999752253293991
0.08999751508235931
0.08999751508235931
0.08999751508235931
0.08999752253293991


In [16]:
print([p for p in model.parameters()])

[Parameter containing:
tensor([[3.0185]], requires_grad=True), Parameter containing:
tensor([7.9957], requires_grad=True)]


In [9]:
# equivalent way of specifiying the same model
class LinearRegression(nn.Module):
    def __init__(self):
        super(LinearRegression, self).__init__()
        self.lin = nn.Linear(1, 1)
        
    def forward(self, x):
        x = self.lin(x)
        return x 
model2 =  LinearRegression()

### Practice (15 - 20 min)
 - Create a new `gen_fake` function that takes returns 2 features (`x1` and `x2`) and one output `y`, where `y` is 0 or 1, based on a linear separation. In other words, you should be able to plot your data like the figure below

![](https://docs.google.com/uc?export=download&id=1sTK7PceuEa46JOn0-kY0KD18Jlncs4Vh)

## Further Reading <a id="reading"></a>


* <a href="https://www.manning.com/books/deep-learning-with-pytorch?query=pytorch"> Deep Learning with PyTorch </a>
* <a href="https://github.com/yanneta/deep-learning-data-institute/blob/master/lesson1-intro.ipynb"> Yannet Interian's GitHub </a>
* https://pytorch.org/docs/stable/index.html
* http://pytorch.org/tutorials/beginner/pytorch_with_examples.html
* https://hsaghir.github.io/data_science/pytorch_starter/